# SafeVision AI — Entrenamiento del modelo de caídas

Este cuaderno entrena un modelo YOLO propio que detecta personas caídas. Se ejecuta en Google Colab con GPU gratuita.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

Pasos: 1) comprobar la GPU, 2) descargar el código, 3) descargar el dataset, 4) entrenar, 5) revisar resultados, 6) descargar el modelo.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi


## 2. Descargar el código del módulo IoT

In [ ]:
import os

if not os.path.exists("/content/SafeVisionAI-IoT"):
    !git clone https://github.com/Luisaparrac/SafeVisionAI-IoT.git /content/SafeVisionAI-IoT
%cd /content/SafeVisionAI-IoT
!pip install -q ultralytics pyyaml roboflow

## 3. Descargar el dataset

En [Roboflow Universe](https://universe.roboflow.com/search?q=class%3Afall) elige un dataset de caídas y pulsa **Download Dataset → formato YOLOv11 → Show download code**. En la celda de abajo cambia solo el workspace, el proyecto y la versión.

También puedes subir un `.zip` en formato YOLO y descomprimirlo en `datasets/`.

**Cómo crear el secreto en Colab:**

1. Abre el panel de secretos con el **ícono de llave**, a la izquierda.
2. Pulsa **Add new secret**.
3. En *Name* escribe `ROBOFLOW_API_KEY` y en *Value* pega tu clave.
4. **Activa el acceso** para este cuaderno.

**Nunca pegues tu clave en el cuaderno ni lo guardes en GitHub con ella.**

In [ ]:
# Replace the workspace, project and version with yours
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("TU_WORKSPACE").project("TU_PROYECTO")
version = project.version(1)
dataset = version.download("yolov11")

DATA_YAML = f"{dataset.location}/data.yaml"

## 4. Entrenar

Parte de `yolo11n.pt`, ya entrenado con el dataset COCO, y lo ajusta a las caídas (*transfer learning*). Con una GPU T4 y unas 3.000 imágenes, 50 épocas tardan alrededor de una hora.

In [ ]:
!python training/train_fall_model.py --data {DATA_YAML} --epochs 50 --imgsz 640 --batch 16 --device 0

## 5. Revisar los resultados

- **Precision:** de las caídas que detecta el modelo, cuántas eran caídas reales.
- **Recall:** de las caídas reales, cuántas detectó.
- **mAP50:** calidad general de la detección (de 0 a 1; mientras más alto, mejor).

In [ ]:
import glob
import os
from IPython.display import Image, display

found = sorted(glob.glob("/content/**/fall_model*/results.png", recursive=True), key=os.path.getmtime)
if not found:
    print("No se encontraron gráficas. Ejecuta primero el paso 4.")
else:
    folder = os.path.dirname(found[-1])
    print(f"Resultados en: {folder}")
    for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
        image = os.path.join(folder, name)
        if os.path.exists(image):
            display(Image(filename=image, width=900))

## 6. Descargar el modelo

Guarda `fall_model.pt` en la carpeta `models/` del módulo IoT y agrega al `.env`:

```
FALL_MODEL=models/fall_model.pt
```

In [ ]:
from google.colab import files
files.download("models/fall_model.pt")